# Neural Pattern Associator (NPA)

Paper: [Within-basket Recommendation via Neural Pattern Associator](https://arxiv.org/abs/2401.16433)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
import os

os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
# os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

device = 'cpu'
print(f"Using device: {device}")

torch.set_num_threads(1)
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

print(f"Torch threads: {torch.get_num_threads()}")

Using device: cpu
Torch threads: 1


In [3]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

from tecd_retail_recsys.data import DataPreprocessor
from tecd_retail_recsys.models.npa import NPATrainer


In [4]:
dp = DataPreprocessor(
    day_begin=1082, 
    day_end=1308, 
    val_days=20, 
    test_days=20, 
    min_user_interactions=1, 
    min_item_interactions=20
)
train_df, val_df, test_df = dp.preprocess()

Starting data preprocessing...
Loading events from t_ecd_small_partial/dataset/small/retail/events
Loaded 236,479,226 total events
Loading items data from t_ecd_small_partial/dataset/small/retail/items.pq
Loaded 250,171 items with features: ['item_id', 'item_brand_id', 'item_category', 'item_subcategory', 'item_price', 'item_embedding']
Merged item features. Data shape: (236479226, 12)
Filtered to 3,758,762 events with action_type='added-to-cart'
After filtering (min_user_interactions=1, min_item_interactions=20): 3,249,972 events, 84,944 users, 30,954 items
Created mappings: 84944 users, 30954 items
Temporal split - Train: days < 1269 (902,543 events), Val: days 1269-1288 (228,339 events), Test: days >= 1289 (223,395 events)
Users in each part (train, val, test) - 7425


In [ ]:
def group_by_baskets(df, max_basket_size=30):
    """Group interactions into baskets by user and session."""
    user_baskets = {}
    user_ids = []
    
    for user_id, group in df.groupby('user_id'):
        if 'session_id' in group.columns:
            baskets = [session_group['item_id'].tolist() 
                      for _, session_group in group.groupby('session_id')]
        else:
            items = group['item_id'].tolist()
            baskets = [items[i:i+max_basket_size] for i in range(0, len(items), max_basket_size)]
        
        user_baskets[user_id] = baskets
        user_ids.append(user_id)
    
    return user_baskets, user_ids

train_user_baskets, train_user_ids = group_by_baskets(train_df)
val_user_baskets, val_user_ids = group_by_baskets(val_df)
test_user_baskets, test_user_ids = group_by_baskets(test_df)

Grouping interactions into baskets...

Basket statistics:
  Train users: 7425
  Val users: 7425
  Test users: 7425


In [22]:
trainer = NPATrainer(
    train_user_baskets,
    # === Architecture ===
    embed_dim=256,           # размерность эмбеддингов
    n_layers=4,              # кол-во слоёв VQA (stacked)
    n_channels=8,            # кол-во параллельных VQA каналов в слое
    codebook_size=64,        # размер кодбука паттернов
    dropout=0.1,
    variant="MC",            # "SC" (squashed-context) или "MC" (multi-context)
    # === Training ===
    lr=3e-4,
    weight_decay=1e-5,
    batch_size=256,
    epochs=20,
    # === Evaluation ===
    holdout_ratio=0.15,       # доля корзин для eval
    target_ratio=0.3,        # последние 20% корзины → ground truth
    eval_k=100,              # NDCG@100
    # === Other ===
    max_basket_size=30,
    seed=42,
    device="auto",           # "cpu", "cuda", "auto"
    verbose=True,
)

Items:         30751
Total baskets: 33863
Train baskets: 28784
Eval baskets:  5079
Device:        cpu
Model params:  31,318,560


In [23]:
trainer.train()

Epoch   1/20 │ loss=10.0510 │ NDCG@100=0.0416 │ Recall@100=0.0855
Epoch   2/20 │ loss=9.7881 │ NDCG@100=0.0465 │ Recall@100=0.0939
Epoch   3/20 │ loss=9.7097 │ NDCG@100=0.0511 │ Recall@100=0.0979
Epoch   4/20 │ loss=9.6146 │ NDCG@100=0.0607 │ Recall@100=0.1093
Epoch   5/20 │ loss=9.5270 │ NDCG@100=0.0652 │ Recall@100=0.1154
Epoch   6/20 │ loss=9.4434 │ NDCG@100=0.0702 │ Recall@100=0.1235
Epoch   7/20 │ loss=9.3612 │ NDCG@100=0.0718 │ Recall@100=0.1285
Epoch   8/20 │ loss=9.3055 │ NDCG@100=0.0736 │ Recall@100=0.1327
Epoch   9/20 │ loss=9.2183 │ NDCG@100=0.0761 │ Recall@100=0.1361
Epoch  10/20 │ loss=9.2029 │ NDCG@100=0.0760 │ Recall@100=0.1354
Epoch  11/20 │ loss=9.1497 │ NDCG@100=0.0772 │ Recall@100=0.1364
Epoch  12/20 │ loss=9.1270 │ NDCG@100=0.0769 │ Recall@100=0.1370
Epoch  13/20 │ loss=9.1127 │ NDCG@100=0.0781 │ Recall@100=0.1386
Epoch  14/20 │ loss=9.0627 │ NDCG@100=0.0792 │ Recall@100=0.1391
Epoch  15/20 │ loss=9.0738 │ NDCG@100=0.0793 │ Recall@100=0.1394
Epoch  16/20 │ loss=9.04

In [25]:
metrics = trainer.evaluate()

Eval  NDCG@100  = 0.0803
Eval  Recall@100 = 0.1411


In [28]:
model_path = "./models/npa/best_model.pt"

checkpoint = {
    "model_state_dict": trainer.model.state_dict(),
    "optimizer_state_dict": trainer.optimizer.state_dict(),
    "scheduler_state_dict": trainer.scheduler.state_dict(),
    "model_config": {
        "num_items": trainer.model.num_items,
        "embed_dim": trainer.model.embed_dim,
        "n_layers": len(trainer.model.layers),
        "n_channels": len(trainer.model.layers[0].channels),
        "codebook_size": trainer.model.layers[0].channels[0].codebook_size,
        "variant": trainer.model.variant,
    },
    "trainer_config": {
        "eval_k": trainer.eval_k,
        "target_ratio": trainer.target_ratio,
        "batch_size": trainer.batch_size,
    },
}
torch.save(checkpoint, model_path)

In [24]:
# инференс для конкретной корзины (аля realtime)
basket = [8591, 4010, 750]
recommendations = trainer.predict(basket, top_k=10)
print("\nTop-10 recommendations:")
for item_id, score in recommendations:
    print(f"  item {item_id:>6d}  score={score:.4f}")


Top-10 recommendations:
  item  18856  score=0.0081
  item  20587  score=0.0068
  item  16561  score=0.0060
  item  30249  score=0.0053
  item   8897  score=0.0051
  item  10104  score=0.0046
  item  14723  score=0.0045
  item  11570  score=0.0043
  item  21151  score=0.0038
  item  27552  score=0.0033


In [30]:
# разные стратегии рекомендаций на подвыборке пользователей

sample_val_user_baskets = {k:v for k,v in val_user_baskets.items() if k in list(val_user_baskets.keys())[:100]}

# Greedy
res_greedy = trainer.evaluate_autoregressive(
    sample_val_user_baskets,
    target_ratio=0.3,
    top_k=100,
    strategy="greedy",
)

# Nucleus sampling
res_nucleus = trainer.evaluate_autoregressive(
    sample_val_user_baskets,
    target_ratio=0.3,
    top_k=100,
    strategy="nucleus",
    top_p=0.9,
    temperature=0.7,
)
# Multirun top-k sampling
res_multi = trainer.evaluate_autoregressive(
    sample_val_user_baskets,
    target_ratio=0.3,
    top_k=100,
    strategy="top_k",
    top_k_sample=200,
    temperature=0.7,
    n_runs=10,
)

# Default (not autoregressive) sampling
res_single = trainer.evaluate_external(sample_val_user_baskets, target_ratio=0.3, top_k=100)

print("\n===== Сравнение =====")
print(f"Default (Not-AR):  NDCG={res_single['ndcg']:.4f}  Recall={res_single['recall']:.4f}")
print(f"AR greedy:         NDCG={res_greedy['ndcg']:.4f}  Recall={res_greedy['recall']:.4f}")
print(f"AR nucleus:        NDCG={res_nucleus['ndcg']:.4f}  Recall={res_nucleus['recall']:.4f}")
print(f"AR multi-run (10): NDCG={res_multi['ndcg']:.4f}  Recall={res_multi['recall']:.4f}")


Strategy:         greedy
Baskets evaluated: 144
NDCG@100:       0.0564
Recall@100:     0.1095
HitRate@100:    0.3958
Strategy:         nucleus
Baskets evaluated: 144
NDCG@100:       0.0349
Recall@100:     0.0897
HitRate@100:    0.3333
Strategy:         top_k (x10 runs)
Baskets evaluated: 144
NDCG@100:       0.0517
Recall@100:     0.1052
HitRate@100:    0.3750
Baskets evaluated: 144
NDCG@100:       0.0577
Recall@100:     0.1157

===== Сравнение =====
Default (Not-AR):  NDCG=0.0577  Recall=0.1157
AR greedy:         NDCG=0.0564  Recall=0.1095
AR nucleus:        NDCG=0.0349  Recall=0.0897
AR multi-run (10): NDCG=0.0517  Recall=0.1052


In [31]:
# evaluation with best strategy
best_strategy_metrics = trainer.evaluate_external(
    val_user_baskets,
    target_ratio=0.3,
    top_k=100
)

Baskets evaluated: 11006
NDCG@100:       0.0647
Recall@100:     0.1306


`Удалось добиться ndcg@100 = 0.065 на задаче within-basket recommendation. Результат ниже, чем у персонализированных подходов, использующих историю покупок. Модель NPA не требует истории пользователя и опирается исключительно на паттерны совместных покупок, выученные из обучающих корзин. Это делает ее применимой в сценарии холодного старта, где персонализированные методы неработоспособны. Обученная модель работает в real-time режиме, адаптируя рекомендации по мере добавления товаров в корзину. Ограничение: модель не учитывает индивидуальные предпочтения пользователя и не способна рекомендовать товары, отсутствующие в обучающей выборке.`